# MutaShield-Net Demo Notebook
**Paper:** A Hybrid Mutation-Guided Adversarial Evolution Framework for Adaptive Web Intrusion Detection Systems

This notebook walks through: dataset loading → model building → training → evaluation → visualization.

In [ ]:
import sys, os
sys.path.insert(0, '..')  # Add project root
import numpy as np
import torch
import matplotlib.pyplot as plt
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 1. Configuration

In [ ]:
from config import *
print('Dataset:', DATASET_NAME)
print('SMOE config:', SMOE)
print('CA-GRT config:', CAGRT)
print('AMFEL config:', AMFEL)

## 2. Dataset Download & Loading
> Place CICIDS2017 CSV files in `data/cicids2017/` before running.

In [ ]:
from dataset import download_cicids2017
download_cicids2017()

In [ ]:
# If data is present, build loaders
try:
    from dataset import build_dataloaders
    train_loader, val_loader, test_loader, scaler, encoder = build_dataloaders()
    print('Loaders ready')
except FileNotFoundError as e:
    print(f'Data not found: {e}')
    print('Using synthetic demo data instead.')
    # Synthetic demo
    import torch
    from torch.utils.data import TensorDataset, DataLoader
    N = 1000; T = 100; d = 40
    X = torch.randn(N, T, d); Y = torch.randint(0, 8, (N,))
    ds = TensorDataset(X, X, Y)
    train_loader = val_loader = test_loader = DataLoader(ds, batch_size=32)

## 3. Model Architecture

In [ ]:
from model import MutaShieldNet
from utils import model_summary

model = MutaShieldNet()
print(model_summary(model))

In [ ]:
# Test forward pass
B, T, d = 4, 100, 40
p = torch.randn(B, T, d)
s = torch.randn(B, T, d)
out = model(p, s)
print(f'Output shape: {out.shape}  (batch x classes)')

## 4. SMOE — Mutation Demo

In [ ]:
from model import SMOE

smoe = SMOE()
x_demo = torch.randn(8, 80)  # 8-sample batch, 80 features
mutants = smoe.generate_pool(x_demo)
print(f'Generated {len(mutants)} mutant variants (one per operator)')
print(f'Mutant shape: {mutants[0].shape}')

# Semantic distance
dists = [(m - x_demo).norm(dim=-1).mean().item() for m in mutants]
plt.bar(range(len(dists)), dists)
plt.xlabel('Operator ID'); plt.ylabel('Mean L2 Distance')
plt.title('Mutation Diversity per Operator')
plt.tight_layout(); plt.show()

## 5. Quick Training (5 epochs demo)

In [ ]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = MutaShieldNet().to(device)
opt    = optim.Adam(model.parameters(), lr=1e-4)

losses = []
for epoch in range(1, 6):
    model.train()
    ep_loss = 0
    for p_seq, s_seq, labels in train_loader:
        p_seq, s_seq, labels = p_seq.to(device), s_seq.to(device), labels.to(device)
        opt.zero_grad()
        loss, _ = model.adversarial_loss(p_seq, s_seq, labels)
        loss.backward()
        opt.step()
        ep_loss += loss.item()
    losses.append(ep_loss)
    print(f'Epoch {epoch}: loss={ep_loss:.4f}')

plt.plot(losses, 'b-o')
plt.xlabel('Epoch'); plt.ylabel('Total Loss')
plt.title('Demo Training Loss')
plt.tight_layout(); plt.show()

## 6. Evaluation Demo

In [ ]:
import torch.nn.functional as F
from utils import compute_metrics

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for p_seq, s_seq, labels in test_loader:
        logits = model(p_seq.to(device), s_seq.to(device))
        preds  = logits.argmax(dim=-1)
        all_preds.append(preds.cpu())
        all_labels.append(labels)

y_pred  = torch.cat(all_preds).numpy()
y_true  = torch.cat(all_labels).numpy()
metrics = compute_metrics(y_true, y_pred)
print('Metrics:')
for k, v in metrics.items():
    print(f'  {k:12s}: {v*100:.2f}%')

## 7. Visualize Results

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.tight_layout(); plt.show()

## 8. Paper Figures
Display the pre-generated figures from the paper.

In [ ]:
from IPython.display import Image, display
import glob

fig_paths = sorted(glob.glob('../figures/*.png'))
for fp in fig_paths:
    print(os.path.basename(fp))
    display(Image(fp, width=700))